In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
import datetime
import joblib
from sklearn.model_selection import GridSearchCV
import os

In [18]:
# 1. LOAD DỮ LIỆU
path = '../../../data/insurance.csv'
if os.path.exists(path):
    print("✅ Đường dẫn ĐÚNG! File đã tìm thấy.")
    df = pd.read_csv(path)
else:
    print("Đường dẫn SAI! Python không thấy file.")

✅ Đường dẫn ĐÚNG! File đã tìm thấy.


In [19]:
# 2. FEATURE ENGINEERING (Tạo biến mới như bạn đã phân tích)
def feature_engineering(df):
    temp_df = df.copy()
    conditions = [
        (temp_df['bmi'] < 18.5),
        (temp_df['bmi'] < 25),
        (temp_df['bmi'] < 30),
        (temp_df['bmi'] >= 30)
    ]
    choices = ['Underweight', 'Normal', 'Overweight', 'Obese']
    temp_df['bmi_category'] = np.select(conditions, choices, default='Normal')
    
    temp_df['obese_smoker'] = ((temp_df['bmi'] >= 30) & (temp_df['smoker'] == 'yes')).astype(int)
    return temp_df

df_fe = feature_engineering(df)

In [20]:
# 3. CHIA TÁCH DỮ LIỆU
X = df_fe.drop('charges', axis=1)
y = df_fe['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:

# 4. ĐỊNH NGHĨA BỘ TIỀN XỬ LÝ (PREPROCESSOR)
def get_preprocess(X):
    categorical_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
    numerical_cols = X.select_dtypes(include=['int64', 'float64', 'number']).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_cols),
            ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
        ]
    )
    return preprocessor

preprocessor = get_preprocess(X_train)

In [22]:
# 5. TIỀN XỬ LÝ VÀ TRAIN MODEL
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # Tạm thời bỏ các tham số gán cứng đi, chỉ giữ lại random_state
    ('model', GradientBoostingRegressor(random_state=42))
])

# Định nghĩa lưới tham số muốn thử nghiệm (nhớ thêm tiền tố 'model__' vì nó nằm trong Pipeline)
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 4, 5]
}

# Khởi tạo GridSearchCV với 5-Fold CV
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

print("⏳ Đang tìm kiếm tham số tối ưu cho Gradient Boosting...")
grid_search.fit(X_train, y_train)

# Lấy ra mô hình xịn nhất
best_pipeline = grid_search.best_estimator_
print(f"✅ Tham số tốt nhất tìm được: {grid_search.best_params_}")

⏳ Đang tìm kiếm tham số tối ưu cho Gradient Boosting...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
✅ Tham số tốt nhất tìm được: {'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 300}


In [23]:
# 6. ĐÁNH GIÁ MÔ HÌNH
y_pred = best_pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("="*50)
print(f"{'KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (GRADIENT BOOSTING)':^50}")
print("="*50)
print(f"R2 Score (Trên tập Test): {r2:>15.4f}")
print(f"MAE (Sai số tuyệt đối): {f'{mae:,.2f}':>22}")
print(f"RMSE (Sai số căn bậc hai): {f'{rmse:,.2f}':>20}")
print("="*50)


   KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (GRADIENT BOOSTING)   
R2 Score (Trên tập Test):          0.8683
MAE (Sai số tuyệt đối):               2,725.05
RMSE (Sai số căn bậc hai):             4,520.91


In [24]:
# 7. TRỰC QUAN HÓA
import os
import joblib
import datetime

metadata = {
    "model_name": "Gradient Boosting Regressor",
    "version": "1.0",
    "date_created": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "metrics": {
        "r2": r2,
        "mae": mae,
        "rmse": rmse
    },
    "features": list(X_train.columns)
}

model_package = {
    "pipeline": best_pipeline,
    "metadata": metadata
}

current_dir = os.path.abspath('')

# Đi ngược lên 2 cấp để ra thư mục MEDICAL-INSURANCE (từ modeling -> notebooks -> root)
project_root = os.path.dirname(os.path.dirname(current_dir))

# Trỏ thẳng vào thư mục models ngoài cùng
save_dir = os.path.join(project_root, 'models')

# Đảm bảo thư mục tồn tại
os.makedirs(save_dir, exist_ok=True)

# Lưu file
joblib_path = os.path.join(save_dir, 'gradient_boosting_regressor_pipeline.joblib')
joblib.dump(model_package, joblib_path)

print(f"\n✅ Đã lưu thành công Full Pipeline và Metadata tại:\n{joblib_path}")


✅ Đã lưu thành công Full Pipeline và Metadata tại:
C:\Users\ASUS\OneDrive\Desktop\Medical-Insurance\notebooks\models\gradient_boosting_regressor_pipeline.joblib
